# 🏥 Clinical Text Classifier — Fine-Tuning DistilBERT on MedNLI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/clinical-text-classifier/blob/main/clinical_text_classifier.ipynb)

**Author:** Kanishka Patel | **Model:** DistilBERT | **Task:** Clinical NLI (3-class)

---

## 🚀 Zero-Install Setup (Google Colab)

This notebook runs **100% in your browser for free** via Google Colab — no Python, Jupyter, or anything else to install.

**Enable free GPU before running (highly recommended):**
1. In the top menu: **Runtime → Change runtime type**
2. Set **Hardware accelerator → T4 GPU**
3. Click **Save**, then run all cells

---

## 🎯 What This Does

Fine-tunes **DistilBERT** on clinical sentence pairs to classify their logical relationship:

| Label | Meaning | Clinical Example |
|---|---|---|
| ✅ **Entailment** | Hypothesis follows from premise | *"BP: 165/100"* → *"Patient has hypertension"* |
| ➖ **Neutral** | No logical connection | *"BP: 165/100"* → *"Patient was discharged"* |
| ❌ **Contradiction** | Hypothesis contradicts premise | *"BP: 165/100"* → *"BP is completely normal"* |

**Stack:** PyTorch · HuggingFace Transformers · Datasets · Evaluate

## Step 1 — Check GPU & Install Packages

In [ ]:
# ── Check GPU availability ────────────────────────────────────────────────────
import subprocess, sys

try:
    gpu_info = subprocess.check_output('nvidia-smi', shell=True).decode()
    print("✅ GPU is available!")
    print(gpu_info[:500])
except Exception:
    print("⚠️  No GPU detected.")
    print("   Training will work on CPU but will be slower.")
    print("   To enable free GPU: Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Install packages (takes ~60 seconds on first run) ─────────────────────────
# Most of these are already on Colab — this just ensures correct versions.
!pip install -q transformers datasets evaluate accelerate scikit-learn seaborn
print("✅ All packages installed!")

## Step 2 — Imports & Configuration

In [ ]:
import os
import numpy as np
import torch
from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Hyperparameters ───────────────────────────────────────────────────────────
MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH       = 256
BATCH_SIZE       = 16
LEARNING_RATE    = 2e-5
NUM_EPOCHS       = 3
WEIGHT_DECAY     = 0.01
OUTPUT_DIR       = "./clinical-classifier-checkpoints"
SEED             = 42

LABEL2ID = {"entailment": 0, "neutral": 1, "contradiction": 2}
ID2LABEL = {0: "entailment", 1: "neutral", 2: "contradiction"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import transformers
print(f"✅ PyTorch:       {torch.__version__}")
print(f"✅ Transformers:  {transformers.__version__}")
print(f"✅ Device:        {device}" + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else " (CPU)"))

## Step 3 — Load Dataset

We include a **built-in clinical dataset** that runs with zero setup.  
Optionally, you can swap in the real **MedNLI** dataset (requires free HuggingFace login + license acceptance).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  OPTION A (Default) — Built-in synthetic clinical dataset               ║
# ║  Runs immediately with no setup or login required.                      ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  OPTION B — Real MedNLI from HuggingFace                               ║
# ║  Requires: huggingface-cli login + accept terms at                      ║
# ║  https://huggingface.co/datasets/bigbio/mednli                         ║
# ║  Then uncomment the load_dataset line below and comment out Option A.   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── OPTION B (uncomment to use real MedNLI) ──────────────────────────────────
# from huggingface_hub import notebook_login
# notebook_login()   # prompts for HF token in Colab
# raw_datasets = load_dataset("bigbio/mednli", name="mednli_bigbio_te")

# ── OPTION A — Built-in clinical dataset (default) ───────────────────────────
synthetic_data = {
    "train": [
        # Entailment (premise → hypothesis logically follows)
        {"premise": "The patient presented with acute chest pain radiating to the left arm and diaphoresis.",
         "hypothesis": "The patient experienced chest discomfort.", "label": "entailment"},
        {"premise": "Blood pressure was recorded at 168/104 mmHg on two separate readings.",
         "hypothesis": "The patient has hypertension.", "label": "entailment"},
        {"premise": "CT scan revealed a 2.1 cm pulmonary nodule in the right lower lobe.",
         "hypothesis": "An abnormality was found in the patient's lung on imaging.", "label": "entailment"},
        {"premise": "Patient reported progressive shortness of breath on exertion for three weeks.",
         "hypothesis": "The patient has breathing difficulties.", "label": "entailment"},
        {"premise": "Serum creatinine was elevated at 3.2 mg/dL with a GFR of 18.",
         "hypothesis": "Kidney function is significantly impaired.", "label": "entailment"},
        {"premise": "EKG showed ST-segment elevation in leads II, III, and aVF.",
         "hypothesis": "The EKG findings are consistent with cardiac abnormality.", "label": "entailment"},
        {"premise": "Patient is a known type 2 diabetic currently on metformin 500mg twice daily.",
         "hypothesis": "The patient takes oral medication for diabetes management.", "label": "entailment"},
        {"premise": "Oxygen saturation dropped to 88% on room air, requiring supplemental O2.",
         "hypothesis": "The patient's oxygen levels were dangerously low.", "label": "entailment"},
        {"premise": "Hemoglobin A1c returned at 9.8%, indicating poor glycemic control.",
         "hypothesis": "The patient's blood sugar is not well controlled.", "label": "entailment"},
        {"premise": "Patient has a documented allergy to penicillin causing anaphylaxis.",
         "hypothesis": "Penicillin is contraindicated for this patient.", "label": "entailment"},
        {"premise": "The patient underwent coronary artery bypass grafting five years ago.",
         "hypothesis": "The patient has a history of cardiac surgery.", "label": "entailment"},
        {"premise": "WBC count was 18,400 cells/μL with a left shift.",
         "hypothesis": "The white blood cell count is elevated, suggesting infection.", "label": "entailment"},
        # Neutral (no logical relationship between premise and hypothesis)
        {"premise": "The patient presented with acute chest pain radiating to the left arm.",
         "hypothesis": "The patient was discharged the following morning.", "label": "neutral"},
        {"premise": "Blood pressure was recorded at 168/104 mmHg.",
         "hypothesis": "The patient has a family history of type 2 diabetes.", "label": "neutral"},
        {"premise": "CT scan revealed a 2.1 cm pulmonary nodule in the right lower lobe.",
         "hypothesis": "The patient's last clinic visit was six months ago.", "label": "neutral"},
        {"premise": "Patient reported shortness of breath on exertion for three weeks.",
         "hypothesis": "The patient works as a high school teacher.", "label": "neutral"},
        {"premise": "Serum creatinine was elevated at 3.2 mg/dL.",
         "hypothesis": "The patient was born in 1978 and is currently 47 years old.", "label": "neutral"},
        {"premise": "EKG showed ST-segment elevation in leads II, III, and aVF.",
         "hypothesis": "The patient had a dental extraction two weeks prior.", "label": "neutral"},
        {"premise": "Patient is a known diabetic on metformin.",
         "hypothesis": "The patient lives alone in a single-story home.", "label": "neutral"},
        {"premise": "Oxygen saturation dropped to 88% on room air.",
         "hypothesis": "The patient reported a mild headache on admission.", "label": "neutral"},
        {"premise": "Hemoglobin A1c returned at 9.8%.",
         "hypothesis": "The patient prefers vegetarian meals.", "label": "neutral"},
        {"premise": "The patient underwent coronary artery bypass grafting five years ago.",
         "hypothesis": "The patient's spouse is a nurse.", "label": "neutral"},
        # Contradiction (hypothesis directly contradicts premise)
        {"premise": "The patient presented with acute chest pain radiating to the left arm.",
         "hypothesis": "The patient denied any chest pain or discomfort at presentation.", "label": "contradiction"},
        {"premise": "Blood pressure was recorded at 168/104 mmHg.",
         "hypothesis": "The patient's blood pressure was completely within normal limits.", "label": "contradiction"},
        {"premise": "CT scan revealed a 2.1 cm pulmonary nodule in the right lower lobe.",
         "hypothesis": "The chest CT scan showed absolutely no abnormalities.", "label": "contradiction"},
        {"premise": "Patient reported shortness of breath on exertion for three weeks.",
         "hypothesis": "The patient has had no respiratory symptoms or complaints.", "label": "contradiction"},
        {"premise": "Serum creatinine was elevated at 3.2 mg/dL.",
         "hypothesis": "All renal function tests were entirely within normal reference ranges.", "label": "contradiction"},
        {"premise": "EKG showed ST-segment elevation in leads II, III, and aVF.",
         "hypothesis": "The EKG was completely normal with no abnormal findings.", "label": "contradiction"},
        {"premise": "Patient is a known diabetic on metformin.",
         "hypothesis": "The patient has no history of diabetes or blood sugar issues.", "label": "contradiction"},
        {"premise": "Oxygen saturation dropped to 88% on room air.",
         "hypothesis": "Oxygen saturation was maintained stably at 99% throughout.", "label": "contradiction"},
        {"premise": "Patient has a documented allergy to penicillin causing anaphylaxis.",
         "hypothesis": "The patient has no known drug allergies.", "label": "contradiction"},
        {"premise": "The patient underwent coronary artery bypass grafting five years ago.",
         "hypothesis": "The patient has never had any cardiac procedure or surgery.", "label": "contradiction"},
        {"premise": "WBC count was 18,400 cells/μL with a left shift.",
         "hypothesis": "Complete blood count results were all within normal range.", "label": "contradiction"},
    ],
    "validation": [
        {"premise": "The patient was intubated and placed on mechanical ventilation due to respiratory failure.",
         "hypothesis": "The patient required ventilatory support.", "label": "entailment"},
        {"premise": "MRI of the brain showed no acute intracranial abnormality.",
         "hypothesis": "The brain MRI was normal.", "label": "entailment"},
        {"premise": "Troponin I peaked at 45 ng/mL, consistent with large myocardial infarction.",
         "hypothesis": "The patient suffered a significant heart attack.", "label": "entailment"},
        {"premise": "The patient was intubated and placed on mechanical ventilation.",
         "hypothesis": "The patient's family was present at the bedside.", "label": "neutral"},
        {"premise": "MRI of the brain showed no acute intracranial abnormality.",
         "hypothesis": "The patient is insured through Medicare.", "label": "neutral"},
        {"premise": "Troponin I peaked at 45 ng/mL.",
         "hypothesis": "The patient had visited a cardiologist two years prior.", "label": "neutral"},
        {"premise": "The patient was intubated and placed on mechanical ventilation.",
         "hypothesis": "The patient was breathing comfortably without any support.", "label": "contradiction"},
        {"premise": "MRI of the brain showed no acute intracranial abnormality.",
         "hypothesis": "A large brain tumor was visible on the MRI scan.", "label": "contradiction"},
        {"premise": "Troponin I peaked at 45 ng/mL, consistent with large myocardial infarction.",
         "hypothesis": "Troponin levels were within normal limits throughout admission.", "label": "contradiction"},
    ],
    "test": [
        {"premise": "Patient underwent emergency appendectomy after imaging confirmed acute appendicitis.",
         "hypothesis": "The patient had an emergency surgical procedure.", "label": "entailment"},
        {"premise": "Echocardiogram revealed a severely reduced ejection fraction of 20%.",
         "hypothesis": "The patient has severely reduced cardiac function.", "label": "entailment"},
        {"premise": "Urinalysis showed >100,000 CFU/mL of E. coli.",
         "hypothesis": "The patient has a urinary tract infection.", "label": "entailment"},
        {"premise": "Patient underwent emergency appendectomy.",
         "hypothesis": "The patient has a history of migraines.", "label": "neutral"},
        {"premise": "Echocardiogram revealed a severely reduced ejection fraction of 20%.",
         "hypothesis": "The patient was born in rural Ohio.", "label": "neutral"},
        {"premise": "Urinalysis showed >100,000 CFU/mL of E. coli.",
         "hypothesis": "The patient is enrolled in a clinical trial.", "label": "neutral"},
        {"premise": "Patient underwent emergency appendectomy after imaging confirmed acute appendicitis.",
         "hypothesis": "The patient did not require any surgical intervention.", "label": "contradiction"},
        {"premise": "Echocardiogram revealed a severely reduced ejection fraction of 20%.",
         "hypothesis": "Cardiac function was entirely preserved on echocardiogram.", "label": "contradiction"},
        {"premise": "Urinalysis showed >100,000 CFU/mL of E. coli.",
         "hypothesis": "Urine culture was negative with no growth detected.", "label": "contradiction"},
    ]
}

def make_hf_dataset(records):
    return Dataset.from_dict({
        "premise":    [r["premise"]    for r in records],
        "hypothesis": [r["hypothesis"] for r in records],
        "label":      [LABEL2ID[r["label"]] for r in records],
    })

raw_datasets = DatasetDict({
    "train":      make_hf_dataset(synthetic_data["train"]),
    "validation": make_hf_dataset(synthetic_data["validation"]),
    "test":       make_hf_dataset(synthetic_data["test"]),
})

print("✅ Dataset loaded!")
print(raw_datasets)
print(f"\nTrain:      {len(raw_datasets['train'])} samples")
print(f"Validation: {len(raw_datasets['validation'])} samples")
print(f"Test:       {len(raw_datasets['test'])} samples")

## Step 4 — Exploratory Data Analysis

In [ ]:
# ── Print sample examples ─────────────────────────────────────────────────────
print("=" * 68)
print("SAMPLE TRAINING EXAMPLES")
print("=" * 68)
for i in range(3):
    ex = raw_datasets["train"][i]
    print(f"\n[{i+1}] Label: {ID2LABEL[ex['label']].upper()}")
    print(f"  Premise:    {ex['premise']}")
    print(f"  Hypothesis: {ex['hypothesis']}")

# ── Label distribution plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#4CAF50", "#2196F3", "#F44336"]

for ax, split in zip(axes, ["train", "validation", "test"]):
    labels = [ID2LABEL[l] for l in raw_datasets[split]["label"]]
    df = pd.Series(labels).value_counts().reindex(["entailment", "neutral", "contradiction"])
    bars = ax.bar(df.index, df.values, color=colors, edgecolor="black", alpha=0.85)
    ax.set_title(f"{split.capitalize()} (n={len(labels)})", fontsize=12, fontweight="bold")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=15)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                str(int(bar.get_height())), ha='center', fontsize=11, fontweight='bold')

plt.suptitle("Label Distribution Across Splits", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 5 — Tokenization

DistilBERT receives the sentence pair as:
```
[CLS]  premise tokens  [SEP]  hypothesis tokens  [SEP]
```
The final `[CLS]` hidden state is fed into the classification head.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# Quick demo of what tokenization looks like
demo = raw_datasets["train"][0]
demo_enc = tokenizer(demo["premise"], demo["hypothesis"], truncation=True, max_length=MAX_LENGTH)
print(f"Premise:        {demo['premise'][:60]}...")
print(f"Hypothesis:     {demo['hypothesis']}")
print(f"Token IDs:      {demo_enc['input_ids'][:12]} ... [{len(demo_enc['input_ids'])} total tokens]")
print(f"Decoded tokens: {tokenizer.convert_ids_to_tokens(demo_enc['input_ids'][:12])}")

# Tokenize full dataset
def tokenize_function(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,  # dynamic padding per batch
    )

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["premise", "hypothesis"],
)

print("\n✅ Tokenization complete!")
print(tokenized_datasets)

## Step 6 — Load DistilBERT Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model:              {MODEL_CHECKPOINT}")
print(f"   Total parameters:  {total_params:,}  (~{total_params/1e6:.0f}M)")
print(f"   Trainable params:  {trainable_params:,}")
print(f"   Device:            {device}")

## Step 7 — Define Metrics

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1  = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

print("✅ Metrics: accuracy + macro F1")

## Step 8 — Train

The HuggingFace `Trainer` handles the full training loop — gradient updates, evaluation per epoch, and automatic best-checkpoint loading.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Hyperparameters
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    # Evaluation & checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    # Logging
    logging_steps=5,
    report_to="none",

    # Mixed precision (auto-enabled on GPU)
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🚀 Starting training...")
train_result = trainer.train()

print("\n" + "=" * 50)
print("TRAINING COMPLETE ✅")
print("=" * 50)
print(f"  Final training loss: {train_result.training_loss:.4f}")
print(f"  Total training time: {train_result.metrics.get('train_runtime', 0):.1f}s")
print(f"  Samples/sec:         {train_result.metrics.get('train_samples_per_second', 0):.1f}")

## Step 9 — Evaluate on Test Set

In [ ]:
test_results = trainer.evaluate(tokenized_datasets["test"])

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"  Accuracy:  {test_results.get('eval_accuracy', 0)*100:.1f}%")
print(f"  Macro F1:  {test_results.get('eval_f1_macro', 0)*100:.1f}%")
print(f"  Loss:      {test_results.get('eval_loss', 0):.4f}")

In [ ]:
# ── Confusion matrix + per-class report ──────────────────────────────────────
preds_output = trainer.predict(tokenized_datasets["test"])
preds        = np.argmax(preds_output.predictions, axis=-1)
true_labels  = preds_output.label_ids
label_names  = ["Entailment", "Neutral", "Contradiction"]

cm = confusion_matrix(true_labels, preds)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_names, yticklabels=label_names,
    linewidths=0.5, linecolor='gray', annot_kws={"size": 14}
)
plt.title("Confusion Matrix — Test Set", fontsize=13, fontweight="bold")
plt.ylabel("True Label", fontsize=11)
plt.xlabel("Predicted Label", fontsize=11)
plt.tight_layout()
plt.show()

print("\nPer-class Classification Report:")
print(classification_report(true_labels, preds, target_names=label_names))

## Step 10 — Inference: Try Your Own Clinical Sentences

In [ ]:
def predict_nli(premise: str, hypothesis: str) -> dict:
    """Classify the NLI relationship between a clinical premise and hypothesis."""
    inputs = tokenizer(
        premise, hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    EMOJI = {"entailment": "✅", "neutral": "➖", "contradiction": "❌"}
    predicted = ID2LABEL[probs.argmax()]
    return {
        "label":      predicted,
        "emoji":      EMOJI[predicted],
        "confidence": float(probs.max()),
        "all_scores": {ID2LABEL[i]: float(probs[i]) for i in range(3)},
    }


# ── Test on custom examples ────────────────────────────────────────────────────
test_cases = [
    ("Patient has a history of congestive heart failure and is on furosemide.",
     "The patient takes diuretic medication.",
     "entailment"),
    ("Chest X-ray showed bilateral infiltrates consistent with pneumonia.",
     "The patient recently traveled to Southeast Asia.",
     "neutral"),
    ("The patient is afebrile with a temperature of 36.5°C.",
     "The patient has a high fever of 39.5°C.",
     "contradiction"),
]

print("=" * 70)
print("INFERENCE EXAMPLES")
print("=" * 70)

for i, (premise, hypothesis, expected) in enumerate(test_cases, 1):
    r = predict_nli(premise, hypothesis)
    match = "✅" if r["label"] == expected else "❌"
    print(f"\n[{i}] {match} Predicted: {r['emoji']} {r['label'].upper()}  (Expected: {expected.upper()})")
    print(f"    Confidence: {r['confidence']*100:.1f}%")
    print(f"    Premise:    {premise}")
    print(f"    Hypothesis: {hypothesis}")
    print(f"    Scores → ", end="")
    for lbl, score in r["all_scores"].items():
        print(f"{lbl}: {score:.3f}", end="  ")
    print()

print("\n" + "=" * 70)
print("TRY YOUR OWN — edit the cell below!")
print("=" * 70)

In [ ]:
# ✏️ Edit these two sentences and run the cell!
my_premise    = "The patient was started on IV vancomycin for MRSA bacteremia."
my_hypothesis = "The patient is receiving antibiotic treatment."

result = predict_nli(my_premise, my_hypothesis)
print(f"{result['emoji']} Prediction: {result['label'].upper()}  (confidence: {result['confidence']*100:.1f}%)")
print(f"   Scores: {result['all_scores']}")

## Step 11 — Save the Model

Save locally within Colab, or push directly to the HuggingFace Hub.

In [ ]:
SAVE_PATH = "./clinical-distilbert-mednli"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Model saved to: {SAVE_PATH}/")
print("   Files:", os.listdir(SAVE_PATH))

# Verify it reloads cleanly
test_reload = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
print("✅ Model reloads successfully — ready for deployment!")

In [ ]:
# ── Download model from Colab to your computer ────────────────────────────────
# This zips the model folder and triggers a browser download.

import shutil
shutil.make_archive("clinical-distilbert-mednli", 'zip', SAVE_PATH)

try:
    from google.colab import files
    files.download("clinical-distilbert-mednli.zip")
    print("✅ Download started! Check your Downloads folder.")
except ImportError:
    print("ℹ️  Not running in Colab. Model saved to:", SAVE_PATH)

In [ ]:
# ── (Optional) Push to HuggingFace Hub ────────────────────────────────────────
# This makes your model publicly available at huggingface.co/YOUR_USERNAME/...

# Step 1: Login (opens a token prompt in Colab)
# from huggingface_hub import notebook_login
# notebook_login()

# Step 2: Push
# REPO_NAME = "kanishka/clinical-distilbert-mednli"
# test_reload.push_to_hub(REPO_NAME)
# tokenizer.push_to_hub(REPO_NAME)
# print(f"✅ Live at: https://huggingface.co/{REPO_NAME}")

print("⬆️  Uncomment the block above to push your model to HuggingFace Hub.")

## Summary

| Item | Details |
|---|---|
| **Task** | Clinical NLI — 3-class sentence pair classification |
| **Base model** | `distilbert-base-uncased` (66M params) |
| **Framework** | PyTorch + HuggingFace Transformers |
| **Optimizer** | AdamW + cosine LR schedule + 10% warmup |
| **Runtime** | Google Colab (free T4 GPU) |
| **Training time** | ~1–2 min on T4 GPU / ~5 min on CPU |

### What to try next
- **Better biomedical model**: `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract` — pre-trained on 14M PubMed abstracts
- **Real MedNLI**: Get free access at [physionet.org](https://physionet.org/content/mednli/) (requires registration)
- **Deploy**: Push to HuggingFace Hub → build a Gradio demo → host for free on HF Spaces
- **Extend**: Clinical NER, ICD-10 coding, de-identification

---
*Built with PyTorch & HuggingFace Transformers · Runs free on Google Colab*